#  Meanies - Movement

In [20]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:60% !important; }</style>"))

In [21]:
from PIL import Image, ImageColor, ImageFont, ImageDraw
resize = lambda img,factor: img.resize((int(img.width * factor), int(img.height * factor)), Image.NEAREST)

## Read in the Formation Data

In [22]:
"""
; The first six bytes select the movement strategy
; for the formation from enemyMovementStrategyLoPtrArray.
.BYTE $01 ; Movement Strategy: Enemy 1
.BYTE $01 ; Movement Strategy: Enemy 2
.BYTE $01 ; Movement Strategy: Enemy 3
.BYTE $01 ; Movement Strategy: Enemy 4
.BYTE $01 ; Movement Strategy: Enemy 5
.BYTE $00 ; Movement Strategy: Enemy 6
; The next six bytes select the inital Y position for
; each enemy.
.BYTE $9C ; Initial Y Position : Enemy 1
.BYTE $6C ; Initial Y Position : Enemy 2
.BYTE $B4 ; Initial Y Position : Enemy 3
.BYTE $84 ; Initial Y Position : Enemy 4
.BYTE $CC ; Initial Y Position : Enemy 5
.BYTE $00 ; Initial Y Position : Enemy 6
; THe delay before adding the next enemy, think of it as
; the spacing between enemies as they enter the screen.
.BYTE $05 ; Delay between spawning enemies.
.BYTE $00 ; Initial X Position of Enemy 1
; The last byte is the sprite to be used for the enemies.
.BYTE $09 ; Sprite Value for Enemies
.BYTE $00 ; End Sentinel
"""
game_data_file = "formation_data.asm"
lines_in_file = open(game_data_file,'r').readlines()

formations = []
strategies, y_positions = [],[]
for l in lines_in_file:
    if l.startswith("enemy") and strategies:
        formations += [(
            strategies,
            y_positions,
            delay,
            initial_x,
            sprite_value
        )]
        strategies, y_positions = [],[]
    if "BYTE" not in l:
        continue
    v = int(l[15:17],16)
    if v == 255: v = 2
    if "Movement Strategy" in l:
        strategies += [v]
    elif "Y Position" in l:
        y_positions += [v]
    elif "Delay" in l:
        delay = v
    elif "Initial X" in l:
        initial_x = v
    elif "Sprite Value" in l:
        sprite_value = "{:02X}".format(v)

formations += [(
    strategies,
    y_positions,
    delay,
    initial_x,
    sprite_value
)]
formations[21]

([38, 39, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], 8, 2, '04')

## Read in the Movement Strategies

In [23]:
game_data_file = "movement_strategies.asm"
lines_in_file = open(game_data_file,'r').readlines()
flatten = lambda l: [e for sublist in l for e in sublist]

strategy = []
movement_strategies = []
for l in lines_in_file:
    if l.startswith("movement") and strategy:
        movement_list = flatten([x[14:].strip().split(',') for x in strategy])
        movement_list = [int(x[1:],16) for x in movement_list]
        movement_strategies += [movement_list]
        strategy = []
    if "BYTE" not in l:
        continue
    strategy += [l]

movement_list = flatten([x[14:].strip().split(',') for x in strategy])
movement_list = [int(x[1:],16) for x in movement_list]
movement_strategies += [movement_list]

In [24]:
movement_strategies[8]

[130, 8, 128, 16, 32, 133, 6, 128, 6, 32, 130, 8, 0, 255]

## Create Images of all movement strategies

In [25]:
def drawEnemyMovementOnCanvas(movement_list, canvas, x_pos, y_pos, meanie_name = "1_MEANIE_08"):
    UP       = 0x08
    DOWN     = 0x04
    FORWARD  = 0x02
    BACKWARD = 0x01

    MAX_VELOCITY = 5
    MIN_VELOCITY = -5
    x_velocity = MIN_VELOCITY
    y_velocity = 0

    img = Image.open(f"meanie_sprites/{meanie_name}.png")
    orig_meanie = resize(img, 1)
    meanie = orig_meanie.copy()

    INITIAL_ALPHA = 140
    cycle_alpha = lambda alpha: alpha+15 if alpha < 245 else INITIAL_ALPHA

    stages = []
    it = iter(movement_list)
    for m in it:
        alpha = INITIAL_ALPHA
        ticks = next(it) & 0x7F
        if m == 0:
            ticks = 15
        for tick in range(0, ticks,2):
            if m & 0x02: # mvt += "forward "
                x_velocity = max(MIN_VELOCITY, x_velocity-1)  
            if m & 0x01: # mvt += "backward "
                x_velocity = min(MAX_VELOCITY, x_velocity+1)  
            if m & 0x04: #mvt += "down "
                y_velocity = min(MAX_VELOCITY, y_velocity+1)  
            if m & 0x08: #mvt += "up "
                y_velocity = max(MIN_VELOCITY, y_velocity-1)
            x_pos += x_velocity
            y_pos += y_velocity

            meanie = orig_meanie.copy()
            meanie2 = meanie.copy()
            alpha = cycle_alpha(alpha)
            meanie2.putalpha(alpha)
            meanie.paste(meanie2, meanie)

            canvas.paste(meanie, (x_pos, y_pos), mask=meanie)
        stages += [canvas.copy()]
    return canvas, stages

In [26]:
!mkdir -p formation_movement/strategies/

In [70]:
import random
from PIL import Image, ImageDraw, ImageFont
from itertools import pairwise

def trim(img):
    t,l,b,r = img.getbbox() # top, left, bottom, right
    t,l,b,r = t - 50, l - 50, b + 50, r + 50
    return img.crop((t,l,b,r))
    
for i, movement_list in enumerate(movement_strategies):
    canvas_height = 200
    canvas_width = 400
    canvas = Image.new('RGBA', (canvas_width , canvas_height))
    x = 380
    y = 100
    sprite_value = f"{random.randint(0x00,0x0F) | 0x10:02X}"

    meanie_name = f"{random.randint(1,15)}_MEANIE_{sprite_value}"
    if i == 17:
        meanie_name = f"1_MEANIE_08"

    canvas,stages = drawEnemyMovementOnCanvas(movement_list, canvas, x, y, meanie_name=meanie_name)
    
    resize(canvas,4).save(f"formation_movement/strategies/strategies_{i}.png")
    trim(resize(canvas,4)).save(f"formation_movement/strategies/strategies_{i}_trimmed.png")
    for j, stage in enumerate(stages):
        resize(stage,4).save(f"formation_movement/strategies/strategies_{i}_{j}.png")


## Draw Diagrams of the Strategies

In [63]:
!mkdir -p formation_movement/strategy_diagrams/

In [78]:
def generateStrategyDiagram(strategy_image, strategy_num):

    img = Image.new('RGBA', (strategy_image.width, strategy_image.height + 50))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"MOVEMENT/STRATEGY/{strategy_num+1:02}"
    label_fnt_size = int(strategy_image.width / 33)
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    draw.text((10, 10), label_text, font=label_fnt, fill="black")

    # Main Sprite image
    img.paste(strategy_image, (10,50), mask=strategy_image)
    return img

In [79]:
for i, movement_list in enumerate(movement_strategies):
    formation_image = Image.open(f"formation_movement/strategies/strategies_{i}.png")
    t,l,b,r = formation_image.getbbox() # top, left, bottom, right
    t,l,b,r = t - 50, l - 50, b + 50, r + 50
    cropped_formation_image = formation_image.crop((t,l,b,r))
    diagram_image = generateStrategyDiagram(cropped_formation_image, i)
    diagram_image.save(f"formation_movement/strategy_diagrams/strategies_{i}.png")

## Draw images of all the formation movements

In [28]:
!mkdir -p formation_movement/formations/

In [29]:
from PIL import Image, ImageDraw, ImageFont
from itertools import pairwise

double_size = lambda img: img.resize((int(img.width * 2), int(img.height * 2)), Image.NEAREST)

for i,formation in enumerate(formations):
    strategies, y_positions, delay, initial_x, sprite_value = formation
    meanie_name = f"{random.randint(1,15)}_MEANIE_{sprite_value}"

    canvas = Image.new('RGBA', (800 , 400))
    x = 700
    py = 0
    pairs = list(zip(y_positions, strategies))
    for j, ((y,s),(ny,ns)) in enumerate(pairwise(pairs)):
        if not s or s >= len(movement_strategies): continue
        if not y: y = 100
        if not ny: ny = 100
        movement_list = movement_strategies[s]
        canvas,_ = drawEnemyMovementOnCanvas(movement_list, canvas, x, y, meanie_name)
        x -= delay * 2 if ny != y else 80
    canvas.save(f"formation_movement/formations/formation_movements_{i}.png")


## Draw the Formation Movements for Each Level

In [30]:
level_formations_raw = """level1EnemyFormationOrder = $C590
        .BYTE $00,$19,$04,$0C,$01,$29,$02,$00
        .BYTE $FF
level2EnemyFormationOrder = $C599
        .BYTE $0C,$2F,$22,$08,$1E,$20,$02,$0C
        .BYTE $FF
level3EnemyFormationOrder = $C5A2
        .BYTE $21,$06,$1F,$1C,$25,$0F,$31,$26
        .BYTE $21,$FF
level4EnemyFormationOrder = $C5AC
        .BYTE $1D,$07,$1A,$0A,$34,$0B,$0D,$31
        .BYTE $1D,$FF
level5EnemyFormationOrder = $C5B6
        .BYTE $04,$0E,$2A,$1C,$37,$32,$23,$17
        .BYTE $2E,$04,$FF
level6EnemyFormationOrder = $C5C1
        .BYTE $29,$1E,$0D,$2C,$0F,$1F,$2D,$2F
        .BYTE $0A,$29,$FF
level7EnemyFormationOrder = $C5CC
        .BYTE $0B,$05,$1C,$0C,$16,$2E,$36,$11
        .BYTE $02,$0A,$0B,$FF
level8EnemyFormationOrder = $C5D8
        .BYTE $07,$19,$03,$17,$24,$1D,$02,$21
        .BYTE $0E,$0D,$07,$FF
level9EnemyFormationOrder = $C5E4
        .BYTE $18,$09,$11,$30,$0A,$35,$0D,$26
        .BYTE $2B,$23,$17,$18,$FF
level10EnemyFormationOrder = $C5F1
        .BYTE $2F,$1B,$11,$25,$2A,$33,$31
        .BYTE $08,$1C,$10,$06,$2F,$FF
level11EnemyFormationOrder = $C5FE
        .BYTE $05,$16,$35,$27,$0D,$22,$0A,$00
        .BYTE $36,$1D,$2F,$19,$05,$FF
level12EnemyFormationOrder = $C60C
        .BYTE $37,$1C,$08,$1E,$2F,$2C,$28,$20
        .BYTE $34,$16,$2D,$1F,$37,$FF
level13EnemyFormationOrder = $C61A
        .BYTE $35,$18,$33,$09,$0B,$2A,$00,$0E
        .BYTE $31,$16,$2C,$29,$37,$35,$FF
level14EnemyFormationOrder = $C629
        .BYTE $36,$2E,$0D,$16,$1B,$1A,$1D,$04
        .BYTE $20,$28,$30,$27,$03,$36,$FF
level15EnemyFormationOrder = $C638
        .BYTE $38,$19,$34,$31,$20,$06,$18,$32
        .BYTE $30,$16,$16,$1F,$0C,$35,$38,$FF
"""

In [31]:
level_formations_list = level_formations_raw.split('\n')[0:-1]
level_formations_list
level_formations = []
for i in range(0,len(level_formations_list),3):
    bs = level_formations_list[i+1][14:].split(',')
    bs += level_formations_list[i+2][14:].split(',')
    ns = [int(x[1:],16) for x in bs]
    level_formations += [ns[:-1]]
level_formations

[[0, 25, 4, 12, 1, 41, 2, 0],
 [12, 47, 34, 8, 30, 32, 2, 12],
 [33, 6, 31, 28, 37, 15, 49, 38, 33],
 [29, 7, 26, 10, 52, 11, 13, 49, 29],
 [4, 14, 42, 28, 55, 50, 35, 23, 46, 4],
 [41, 30, 13, 44, 15, 31, 45, 47, 10, 41],
 [11, 5, 28, 12, 22, 46, 54, 17, 2, 10, 11],
 [7, 25, 3, 23, 36, 29, 2, 33, 14, 13, 7],
 [24, 9, 17, 48, 10, 53, 13, 38, 43, 35, 23, 24],
 [47, 27, 17, 37, 42, 51, 49, 8, 28, 16, 6, 47],
 [5, 22, 53, 39, 13, 34, 10, 0, 54, 29, 47, 25, 5],
 [55, 28, 8, 30, 47, 44, 40, 32, 52, 22, 45, 31, 55],
 [53, 24, 51, 9, 11, 42, 0, 14, 49, 22, 44, 41, 55, 53],
 [54, 46, 13, 22, 27, 26, 29, 4, 32, 40, 48, 39, 3, 54],
 [56, 25, 52, 49, 32, 6, 24, 50, 48, 22, 22, 31, 12, 53, 56]]

In [32]:
!mkdir -p formation_movement/level_formations/

In [33]:
i =10
f"{i:02X}"

'0A'

In [34]:
from PIL import Image, ImageDraw, ImageFont
from itertools import pairwise

for level,formations_for_level in enumerate(level_formations):
    for i,formation_index in enumerate(formations_for_level):
        formation = formations[formation_index]
        strategies, y_positions, delay, initial_x, sprite_value = formation
        sprite_value = f"{int(sprite_value,16) | 0x10:02X}"
        meanie_name = f"{level+1}_MEANIE_{sprite_value}"
        canvas = Image.new('RGBA', (800 , 400))
        x = 700
        py = 0
        pairs = list(zip(y_positions, strategies))
        for j, ((y,s),(ny,ns)) in enumerate(pairwise(pairs)):
            if not s or s >= len(movement_strategies): continue
            if not y: y = 100
            if not ny: ny = 100
            movement_list = movement_strategies[s-1]
            canvas,_ = drawEnemyMovementOnCanvas(movement_list, canvas, x, y, meanie_name)
            x -= delay * 2 if ny != y else 10
        resize(canvas,4).save(f"formation_movement/level_formations/{level+1}_formation_movement_{i}_{formation_index}.png")


## Draw Diagrams of the Formation Movements for Each Level

In [77]:
level_names = [None, "Zinc","Lead","Copper","Silver","Iron","Gold","Platinum","Tungsten",
               "Iridon","Kallisto","Tri-alloy","Quadmium","Ergonite","Galactium","Uridium"]

def generateLevelFormationDiagram(formation_image, level, formation_num):

    img = Image.new('RGBA', (formation_image.width, formation_image.height + 50))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"{level+1:02}/{level_names[level+1].upper()}/FORMATION/{formation_num+1:02}"
    label_fnt_size = int(formation_image.width / 33)
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    draw.text((10, 10), label_text, font=label_fnt, fill="black")

    # Main Sprite image
    img.paste(formation_image, (10,50), mask=formation_image)
    return img

In [36]:
!mkdir -p formation_movement/level_formation_diagrams

In [76]:
for level,formations_for_level in enumerate(level_formations):
    for i,formation_index in enumerate(formations_for_level):
        formation_image = Image.open(f"formation_movement/level_formations/{level+1}_formation_movement_{i}_{formation_index}.png")
        t,l,b,r = formation_image.getbbox() # top, left, bottom, right
        t,l,b,r = t - 50, l - 50, b + 50, r + 50
        cropped_formation_image = formation_image.crop((t,l,b,r))
        diagram_image = generateLevelFormationDiagram(cropped_formation_image, level, i)
        diagram_image.save(f"formation_movement/level_formation_diagrams/{level+1}_formation_movement_{i}_{formation_index}.png")

# Scratchpad